# NEST Simulation from SONATA

This notebook demonstrates how to:
1. Generate a SONATA simulation campaign for a NEST circuit using `NestCircuitSimulationScanConfig`
2. Run the simulation using the NEST simulator via `NestSimulationFromSonataTask`
3. Load and plot the results

**Prerequisites:**
- `nest-simulator` (install via conda-forge: `mamba install -c conda-forge nest-simulator`)
- Run `convert_new_allen_v1_to_libsonata_compatible.ipynb` first to generate the libsonata-compatible circuit config

In [ ]:
import json
from pathlib import Path

CIRCUIT_DIR = Path("../../../obi-output/nest_core_nll_7/circuit")
circuit_config_path = CIRCUIT_DIR / "circuit_config.json"

OUTPUT_ROOT = Path("../../../obi-output/nest_core_nll_7")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

## 1. Generate the SONATA simulation campaign

Use `NestCircuitSimulationScanConfig` with the core_nll_7 circuit.
This restricts blocks to NEST-compatible types and produces a
`simulation_config.json` with `target_simulator: "NEST"`.

In [ ]:
from obi_one.core.info import Info
from obi_one.scientific.blocks.neuron_sets.specific import AllNeurons
from obi_one.scientific.blocks.recording import SomaVoltageRecording
from obi_one.scientific.blocks.stimuli.stimulus import ConstantCurrentClampSomaticStimulus
from obi_one.scientific.blocks.timestamps.single import SingleTimestamp
from obi_one.scientific.library.circuit import Circuit
from obi_one.scientific.tasks.generate_simulations.config.nest_circuit import (
    NestCircuitSimulationScanConfig,
)

In [ ]:
SIM_LENGTH_MS = 500.0

circuit = Circuit(name="core_nll_7", path=str(circuit_config_path.absolute()))
print(f"Circuit loaded: {circuit.default_population_name} ({circuit.sonata_circuit.nodes['v1'].size} neurons)")

sim_conf = NestCircuitSimulationScanConfig.empty_config()

# Info
info = Info(
    campaign_name="NEST core_nll_7 demo",
    campaign_description="50 ms NEST simulation of core_nll_7 V1 circuit",
)
sim_conf.set(info, name="info")

# Timestamps
stim_time = SingleTimestamp(start_time=10.0)
sim_conf.add(stim_time, name="stim_onset")

# Neuron sets
all_neurons = AllNeurons()
sim_conf.add(all_neurons, name="AllNeurons")

# Stimulus: constant current 0.05 nA from 10-40 ms
current_stim = ConstantCurrentClampSomaticStimulus(
    timestamps=stim_time.ref,
    neuron_set=all_neurons.ref,
    amplitude=200.0,
    duration=500.0,
)
sim_conf.add(current_stim, name="ConstantCurrent")

# Recording: soma voltage for the full simulation
voltage_rec = SomaVoltageRecording(
    neuron_set=all_neurons.ref,
    dt=1.0,
)
# sim_conf.add(voltage_rec, name="SomaVoltage")

# Initialize
initialize = NestCircuitSimulationScanConfig.Initialize(
    circuit=circuit,
    node_set=all_neurons.ref,
    simulation_length=SIM_LENGTH_MS,
    v_init=-70.0,
)
sim_conf.set(initialize, name="initialize")

validated = sim_conf.validated_config()
print("Config validated.")

In [ ]:
from obi_one.core.run_tasks import run_task_for_single_configs
from obi_one.core.scan_generation import GridScanGenerationTask

campaign_dir = str(OUTPUT_ROOT / "campaign")
grid_scan = GridScanGenerationTask(form=validated, output_root=campaign_dir)
grid_scan.multiple_value_parameters(display=True)
grid_scan.execute()

# Run GenerateSimulationTask for each coordinate to produce simulation_config.json
run_task_for_single_configs(single_configs=grid_scan.single_configs)

single_configs = grid_scan.single_configs
print(f"\nGenerated {len(single_configs)} simulation coordinate(s)")

# Path to the generated simulation_config.json
coord_root = Path(single_configs[0].coordinate_output_root)
sim_config_path = coord_root / "simulation_config.json"
print(f"Simulation config: {sim_config_path}")

In [ ]:
with sim_config_path.open() as f:
    generated_config = json.load(f)

print(f"target_simulator: {generated_config['target_simulator']}")
print(f"tstop: {generated_config['run']['tstop']} ms")
print(f"inputs: {list(generated_config.get('inputs', {}).keys())}")
print(f"reports: {list(generated_config.get('reports', {}).keys())}")

## 2. Run the NEST simulation

Feed the generated `simulation_config.json` to `NestSimulationFromSonataTask`.

In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

In [ ]:
from obi_one.core.path import NamedPath
from obi_one.scientific.tasks.nest_simulation_from_sonata.config import (
    NestSimulationFromSonataSingleConfig,
)
from obi_one.scientific.tasks.nest_simulation_from_sonata.task import (
    NestSimulationFromSonataTask,
)

nest_run_config = NestSimulationFromSonataSingleConfig.empty_config()

nest_init = NestSimulationFromSonataSingleConfig.Initialize(
    simulation_config_path=NamedPath(
        name="core_nll_7 generated config",
        path=str(sim_config_path.absolute()),
    ),
    nest_model="iaf_psc_alpha",
)
nest_run_config.set(nest_init, name="initialize")
nest_run_config.scan_output_root = str(coord_root)
nest_run_config.coordinate_output_root = str(coord_root)

In [ ]:
task = NestSimulationFromSonataTask(config=nest_run_config)
task.execute()

## 3. Analyze results

In [ ]:
import h5py
import matplotlib.pyplot as plt
import numpy as np

spike_file = coord_root / "nest_output" / "spikes.h5"
voltage_file = coord_root / "nest_output" / "voltage.h5"

In [ ]:
with h5py.File(spike_file, "r") as f:
    for pop_name in f["spikes"]:
        timestamps = f["spikes"][pop_name]["timestamps"][:]
        node_ids = f["spikes"][pop_name]["node_ids"][:]
        n_unique = len(np.unique(node_ids))
        print(f"Population '{pop_name}': {len(timestamps)} spikes from {n_unique} neurons")

        fig, ax = plt.subplots(figsize=(12, 4))
        ax.scatter(timestamps, node_ids, s=0.3, alpha=0.5, c="black")
        ax.set_xlabel("Time (ms)")
        ax.set_ylabel("Neuron ID")
        ax.set_title(f"Spike raster - {pop_name}")
        ax.set_xlim(0, SIM_LENGTH_MS)
        plt.tight_layout()
        plt.show()

In [ ]:
if voltage_file.exists():
    with h5py.File(voltage_file, "r") as f:
        for report_name in f["reports"]:
            grp = f["reports"][report_name]
            senders = grp["senders"][:]
            timestamps = grp["timestamps"][:]
            data = grp["data"][:]

            unique_neurons = np.unique(senders)[:5]
            fig, ax = plt.subplots(figsize=(12, 4))
            for nid in unique_neurons:
                mask = senders == nid
                ax.plot(timestamps[mask], data[mask], label=f"Neuron {nid}", alpha=0.8)
            ax.set_xlabel("Time (ms)")
            ax.set_ylabel("Membrane potential (mV)")
            ax.set_title(f"Voltage traces - {report_name}")
            ax.legend(loc="upper right", fontsize=8)
            plt.tight_layout()
            plt.show()
else:
    print("No voltage output file found.")